In [1]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
from datetime import datetime
from zoneinfo import ZoneInfo

In [2]:
# Step 1: Load the Dataset
apps_df = pd.read_csv(r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\apps.csv")
reviews_df = pd.read_csv(r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\user_reviews.csv")

In [3]:
apps_df.head()

,Unnamed: 0,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [8]:
# Remove unwanted rows
apps_df = apps_df[apps_df['App'].notna()]

# Convert Installs to numeric
apps_df['Installs'] = (
    apps_df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

apps_df['Installs'] = pd.to_numeric(
    apps_df['Installs'], 
    errors='coerce'
)

# Convert Price to numeric
apps_df['Price'] = (
    apps_df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
)

apps_df['Price'] = pd.to_numeric(
    apps_df['Price'], 
    errors='coerce'
)

# Convert Size to MB
def convert_size(size):
    size = str(size)
    
    if 'M' in size:
        return float(size.replace('M', ''))
    
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    
    else:
        return np.nan

apps_df['Size_MB'] = apps_df['Size'].apply(convert_size)

# Convert Android Version
apps_df['Android Version'] = pd.to_numeric(
    apps_df['Android Ver'].astype(str).str.extract(r'(\d+\.\d+)')[0],
    errors='coerce'
)

# Revenue
apps_df['Revenue'] = apps_df['Installs'] * apps_df['Price']

apps_df.head()

,Unnamed: 0,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Android Version,Revenue
0,0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,NaN,4.0,0.0
1,1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,NaN,4.0,0.0
2,2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,NaN,4.0,0.0
3,3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,NaN,4.2,0.0
4,4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,NaN,4.4,0.0


In [9]:
filtered_df = apps_df[
    (apps_df['Installs'] >= 10000) &
    (apps_df['Revenue'] >= 10000) &
    (apps_df['Android Version'] > 4.0) &
    (apps_df['Size_MB'] > 15) &
    (apps_df['Content Rating'] == 'Everyone') &
    (apps_df['App'].str.len() <= 30)
].copy()

filtered_df.shape

(0, 17)

In [10]:
filtered_df = apps_df[
    (apps_df['Installs'] >= 10000) &
    (apps_df['Android Version'] > 4.0) &
    (apps_df['Size_MB'] > 15) &
    (apps_df['Content Rating'] == 'Everyone') &
    (apps_df['App'].str.len() <= 30) &
    (
        (apps_df['Type'] == 'Free') |
        (
            (apps_df['Type'] == 'Paid') &
            (apps_df['Revenue'] >= 10000)
        )
    )
].copy()

filtered_df.shape

(0, 17)

In [56]:
def get_android_version(version):

    if pd.isna(version):
        return np.nan

    match = re.search(
        r'(\d+(?:\.\d+)?)',
        str(version)
    )

    if match:
        return float(match.group(1))

    return np.nan


apps_df['Android_Version_Num'] = apps_df['Android Ver'].apply(
    get_android_version
)

apps_df[['Android Ver', 'Android_Version_Num']].head()

,Android Ver,Android_Version_Num
0,4.0.3 and up,4.0
1,4.0.3 and up,4.0
2,4.0.3 and up,4.0
3,4.2 and up,4.2
4,4.4 and up,4.4


In [11]:
top_3_categories = (
    filtered_df['Category']
    .value_counts()
    .head(3)
    .index
    .tolist()
)

top_3_categories

[]

In [12]:
top3_df = filtered_df[
    filtered_df['Category'].isin(top_3_categories)
].copy()

top3_df[['App', 'Category', 'Type', 'Installs', 'Revenue']].head()

,App,Category,Type,Installs,Revenue


In [13]:
summary = (
    top3_df
    .groupby(['Category', 'Type'])
    .agg(
        Average_Installs=('Installs', 'mean'),
        Average_Revenue=('Revenue', 'mean')
    )
    .reset_index()
)

summary

,Category,Type,Average_Installs,Average_Revenue


In [21]:


def show_chart_only_between_1pm_2pm():
    
    # Get current Indian time
    current_time = datetime.now(ZoneInfo("Asia/Kolkata"))
    
    current_minutes = current_time.hour * 60 + current_time.minute
    
    # 1:00 PM = 780 minutes
    # 2:00 PM = 840 minutes
    start_time = 13 * 60
    end_time = 14 * 60
    
    if start_time <= current_minutes < end_time:
        
        fig.show()
        
    else:
        
        print("This graph is available only between 1:00 PM and 2:00 PM IST.")

In [22]:
show_chart_only_between_1pm_2pm()

This graph is available only between 1:00 PM and 2:00 PM IST.
